# 2.11 Agents and workflows: chaining, routing, parallelization, agents

Step 2, lesson 11 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287796) (Anthropic Academy).
Exam domain: **1 Agentic Architecture & Orchestration**, the biggest one (27%). Step 13 of the path (the Agent SDK) builds on this.

What this notebook covers:

1. Workflows vs agents, and when to use which
2. Prompt chaining, with a check in code between the steps
3. Routing, with explicit criteria and a structured label
4. Parallelization: independent sub-tasks at the same time, then aggregation
5. Evaluator–optimizer: generate, critique, revise, with a hard round limit
6. An agent: a tool loop where Claude chooses the steps, inspecting the environment before and after acting
7. Coordinators and subagents, stop conditions, and when not to build an agent

> **Model used here:** `claude-sonnet-5-5`, for about 18 short requests in total. The workflow steps use `effort: "low"`, because each step is simple: that's part of the point of splitting a task.

## 1. Workflows vs agents

Anthropic's article [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) draws the line this course follows:

- A **workflow** runs model calls along **paths written in code**. You decide the steps; Claude does each one.
- An **agent** lets Claude **decide the steps**: it runs in a loop, choosing which tool to call next from what it has seen so far, until the task is done.

| | Workflow | Agent |
|---|---|---|
| Steps known in advance? | yes | no, they depend on what it finds |
| Predictability, testing | high: same path every time | lower: the path varies |
| Cost and latency | bounded | open-ended (cap it) |
| Fits | known processes: draft → check → fix, classify → answer | open problems: debugging, research, multi-step tasks in a changing environment |

The advice that runs through the article and the exam guide: **start with the simplest thing that works** (often one well-prompted call with the right context), move to a workflow when a task splits into fixed steps, and build an agent only when the steps really can't be known in advance.

## Setup

In [1]:
import json
import textwrap
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()
client = Anthropic()
model = "claude-sonnet-5-5"


def text_of(message):
    """Return the text of the first text block in a response."""
    return next(block.text for block in message.content if block.type == "text")


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, max_tokens=1000, **kwargs):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages, **kwargs}
    if system:
        params["system"] = system
    return text_of(client.messages.create(**params))


def ask(prompt, system=None, max_tokens=600, effort="low"):
    """One self-contained step of a workflow: a single user message in, text out."""
    params = {"model": model, "max_tokens": max_tokens, "output_config": {"effort": effort},
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        params["system"] = system
    response = client.messages.create(**params)
    if response.stop_reason == "max_tokens":          # a cut-off answer must not pass silently to the next step
        print(f"[warning: reply cut off at max_tokens={max_tokens}]")
    return text_of(response)

## 2. Prompt chaining

**Chaining** splits a task into a sequence of calls, each working on the output of the previous one. Each prompt has one job, so each is easier to get right and to test. Between the steps, plain code can check the intermediate result (a *gate*) and stop or branch.

The course's example is writing an article and then fixing it in a second call. A long list of style rules in the first prompt tends to get some rules missed; a dedicated revision step with the rules as an explicit checklist is more reliable. Then code checks the rules it *can* check (length, banned words), because a rule that must always hold is best enforced in code.

In [2]:
facts = ("Nimbus Home app, version 4.0. New: schedule robot vacuum cleaning by room and time. "
         "Available on iOS and Android from 3 March. Free for all owners of Nimbus vacuums.")

# Step 1: draft. One job: get the content right.
draft = ask(f"Write a short announcement for our customers.\n\n<facts>{facts}</facts>", max_tokens=500)

# Step 2: revise against explicit rules. One job: apply the checklist.
RULES = """- At most 80 words.
- No hype words: revolutionary, game-changing, amazing, ultimate, seamless.
- No emojis and no exclamation marks.
- Mention the date (3 March) and that it's free.
- End with one sentence telling the reader what to do next."""
final = ask(f"""Revise the announcement so it follows every rule. Reply with the revised text only.

<rules>
{RULES}
</rules>

<announcement>
{draft}
</announcement>""", max_tokens=500)


# Gate in code: the rules that code can check exactly.
def check(text):
    banned = ["revolutionary", "game-changing", "amazing", "ultimate", "seamless"]
    problems = []
    if len(text.split()) > 80:
        problems.append(f"{len(text.split())} words")
    problems += [f"uses '{w}'" for w in banned if w in text.lower()]
    if "!" in text:
        problems.append("exclamation mark")
    if "3 march" not in text.lower():
        problems.append("date missing")
    return problems


print(f"DRAFT ({len(draft.split())} words), problems: {check(draft) or 'none'}\n{draft}\n")
print(f"FINAL ({len(final.split())} words), problems: {check(final) or 'none'}\n{final}")

DRAFT (118 words), problems: ['118 words']
**Introducing Nimbus Home 4.0: Cleaning on Your Schedule**

We're excited to announce version 4.0 of the Nimbus Home app, with a new feature that gives you more control over your clean home.

**What's new:** You can now schedule your robot vacuum to clean specific rooms at specific times. Have it tackle the kitchen after breakfast, the living room while you're at work, or the bedrooms on weekends. It's up to you.

**Availability:** The update arrives on iOS and Android on 3 March.

**Cost:** It's free for all owners of Nimbus vacuums. Just update your app to get started.

Thank you for being part of the Nimbus family. We hope the new scheduling makes your days a little easier.

FINAL (78 words), problems: none
**Nimbus Home 4.0: Cleaning on Your Schedule**

Version 4.0 of the Nimbus Home app lets you schedule your robot vacuum to clean specific rooms at specific times. Have it tackle the kitchen after breakfast, the living room while you're at

If the gate still found a problem, the code could loop back once more or flag the text for a person. The point is where each rule is enforced: the model applies the checklist, code verifies what is verifiable.

## 3. Routing

**Routing** classifies the input first, then sends it down a path built for that kind of input: a specialised prompt, a different tool set, a cheaper model for easy cases. The classifier is one cheap call; the dispatch afterwards is ordinary code.

Two things make the classification reliable:

- **Explicit criteria** for every category, with the tricky boundary cases spelled out, instead of bare labels.
- A **structured label**: with structured outputs (`client.messages.parse` with a Pydantic model), the category is guaranteed to be one of the allowed values, so the dispatch code can't receive "Billing issue (probably)".

> **Changed since the course.** The course asks for the category as plain text and parses it. Structured outputs now guarantee the format. (Exam questions may still describe forcing a tool call to get structured output; on Sonnet 5.5 a forced `tool_choice` returns a 400, so use structured outputs or `strict` tools.)

In [3]:
class Route(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    reason: str


ROUTER_PROMPT = """Classify the customer message into exactly one category.

- billing: charges, refunds, invoices, prices, payment methods. A refund request is billing even if the product is broken.
- technical: the robot or the app doesn't work as expected (errors, connection, cleaning problems) and the customer wants it fixed.
- account: login, password, email change, deleting the account, privacy settings.
- other: anything else, including general questions and feedback.

<message>{message}</message>"""

SPECIALISTS = {
    "billing": "You handle billing questions for Nimbus Home. Refunds are possible within 30 days. Be brief and concrete.",
    "technical": "You are Nimbus Home technical support. Give at most three short troubleshooting steps, most likely fix first, in under 120 words.",
    "account": "You handle account questions. Never ask for passwords. Point to Settings > Account for self-service.",
    "other": "You answer general questions about Nimbus Home products briefly and politely.",
}


def route(message):
    response = client.messages.parse(
        model=model, max_tokens=300, output_config={"effort": "low"}, output_format=Route,
        messages=[{"role": "user", "content": ROUTER_PROMPT.format(message=message)}],
    )
    return response.parsed_output


tickets = [
    "My vacuum stopped mid-clean with a red light and the app says error E7. It's two weeks old. What should I do?",
    "The vacuum is broken after a week and I just want my money back, not a repair.",
]
routes = [route(t) for t in tickets]
for ticket, r in zip(tickets, routes):
    print(f"{r.category:9} <- {ticket}\n          ({r.reason})\n")

# Dispatch in code: the first ticket goes to its specialist prompt.
print("ANSWER:", ask(tickets[0], system=SPECIALISTS[routes[0].category], max_tokens=800))

technical <- My vacuum stopped mid-clean with a red light and the app says error E7. It's two weeks old. What should I do?
          (The customer reports the robot vacuum stopped mid-clean with a red light and app error E7, and wants help fixing it. This is a device malfunction, not a refund or billing request.)

billing   <- The vacuum is broken after a week and I just want my money back, not a repair.
          (The customer is requesting a refund rather than a repair; per the rules, a refund request is billing even if the product is broken.)



ANSWER: I don't have Nimbus's official E7 definition in front of me, so these are the most common fixes for a red-light stop mid-clean:

1. **Check for blockages.** Turn the vacuum over and clear hair or debris from the main brush, side brush, and wheels. Make sure the wheels spin freely and the bumper isn't stuck.
2. **Reseat the bin and filter.** Remove the dustbin, empty it, and snap it back in firmly. Confirm the filter is dry and seated correctly.
3. **Restart it.** Place it on the dock, hold the power button for 10 seconds to reset, then start a short clean.

If E7 returns, contact us with your serial number. At two weeks old, it should be covered under warranty for repair or replacement.


The second ticket mentions a broken product but asks for a refund; the boundary rule in the criteria ("a refund request is billing even if the product is broken") is what sends it to billing.

**Exam angle (Domains 1 and 4).** When a router misroutes, the proportionate first fix is to make the **criteria explicit** (definitions, boundary cases, a few examples), not to train a separate classifier or add another model layer. And if the "routing" is only about which *tool* Claude should call, better **tool descriptions** usually fix it without a router at all.

## 4. Parallelization

When sub-tasks don't depend on each other, run them **at the same time** and combine the results. Two variants:

- **Sectioning**: split the task into independent parts (here: three reviewers, each looking at one aspect), each with a focused prompt.
- **Voting**: run the same task several times and combine the answers, for more confidence (for example, flag content if 2 of 3 reviews flag it).

The API calls are independent HTTP requests, so a thread pool is enough. Each sub-task gets only the context it needs; one more call aggregates.

In [4]:
idea = ("Nimbus wants to add a camera to its robot vacuums so the app can show a live view of the home "
        "while the owner is away, and use the images to recognise obstacles such as cables and socks.")

ASPECTS = {
    "customer value": "Assess only the value for customers: who wants this and why.",
    "technical risk": "Assess only the technical risks and costs of building it.",
    "privacy and legal": "Assess only privacy, security and legal risks (EU customers).",
}


def review(aspect):
    start = time.perf_counter()
    text = ask(f"{ASPECTS[aspect]} Answer in exactly 3 bullet points of at most 25 words each.\n\n<idea>{idea}</idea>",
               max_tokens=800)
    return aspect, text, time.perf_counter() - start


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(ASPECTS)) as pool:
    reviews = list(pool.map(review, ASPECTS))
wall = time.perf_counter() - start

for aspect, text, seconds in reviews:
    print(f"--- {aspect} ({seconds:.1f} s)\n{text}\n")
print(f"wall time {wall:.1f} s vs {sum(r[2] for r in reviews):.1f} s if run one after another\n")

combined = "\n\n".join(f'<review aspect="{a}">\n{t}\n</review>' for a, t, _ in reviews)
print("--- recommendation")
print(ask(f"Here are three independent reviews of a product idea.\n\n{combined}\n\n"
          "Weigh them and give a go / no-go / go-with-conditions recommendation in at most 4 sentences.",
          max_tokens=400))

--- customer value (2.3 s)
- **Who wants it:** Pet owners, parents, and frequent travelers want to check on their home remotely, confirming the vacuum ran properly and glimpsing pets, kids, or deliveries.

- **Why obstacle recognition matters:** Customers with cluttered floors hate vacuums tangling on cables or swallowing socks. Camera-based avoidance means fewer rescues, fewer interruptions, and more trust in unattended cleaning.

- **Value limits:** Many buyers will worry about privacy, since a roaming camera in bedrooms feels invasive. Value depends on clear local processing, visible off controls, and strong security.

--- technical risk (3.0 s)
- **Security and privacy engineering:** Live video demands end-to-end encryption, secure boot, and hardened cloud and authentication. A breach could expose private homes, so audits, penetration testing, and ongoing patching raise costs.

- **Hardware and compute constraints:** Adding camera, processor, and memory raises bill-of-materials cos

**Go-with-conditions.** The reviews agree there is real customer value in remote home checks and camera-based obstacle avoidance, but that value depends on trust, and the product carries heavy security, privacy, and cost burdens. Proceed only if the design makes privacy a core feature: on-device obstacle processing, a physical camera shutter and recording indicator, end-to-end encryption, secure boot and signed updates, and a DPIA with GDPR, Cyber Resilience Act, and Radio Equipment Directive compliance built in from the start. Before committing to full development, validate that the added bill-of-materials, cloud, and ongoing security-maintenance costs fit a viable price point and that obstacle detection performs reliably in low light, and treat live remote video as optional or phase-two if those conditions can't be met.


> **Gotcha hit while building this notebook.** The first run asked for "3 bullet points" with `max_tokens=400` and no length limit. All three reviews stopped mid-sentence (`stop_reason` was `"max_tokens"`), and so did the routing answer above; nothing in the code noticed, and only the aggregating call remarked that its input was cut off. A truncated step that flows silently into the next one is a reliability bug. The fix is twofold: each prompt states a length the step can meet, and `ask` now checks `stop_reason` and warns (a production workflow would retry with more room or stop).

**Exam angle (Domain 1).** This is the shape of a **coordinator with parallel subagents**: split work into independent parts, give each worker a focused prompt with exactly the context it needs (workers don't see each other's context or the coordinator's history), run them concurrently, then aggregate. If the parts depend on each other, run them in sequence instead.

## 5. Evaluator–optimizer

One call **generates**, another **evaluates** against explicit criteria and returns feedback, and the generator revises with that feedback. It's a loop, so it needs a stop condition in code: stop when the evaluator passes the result **or** after a fixed number of rounds (here 2), whichever comes first.

Criteria that code can check (word count) are checked in code; the evaluator model handles judgment calls (unsupported claims, tone).

In [5]:
class Evaluation(BaseModel):
    passed: bool
    issues: list[str]


product = ("Nimbus Buddy kids' smartwatch: GPS location shared with parents' app, SOS button that calls two "
           "saved numbers, battery lasts about 2 days, water resistant to 1 metre, ages 6-12, 89 euros.")
CRITERIA = """1. Mentions the battery life and the price.
2. Makes no claim the facts don't support (for example "keeps your child safe", "guaranteed", "100%").
3. Speaks to parents in plain language, without technical jargon."""
MAX_WORDS = 60


def evaluate(text):
    issues = [f"{len(text.split())} words; the limit is {MAX_WORDS}"] if len(text.split()) > MAX_WORDS else []
    verdict = client.messages.parse(
        model=model, max_tokens=500, output_config={"effort": "low"}, output_format=Evaluation,
        messages=[{"role": "user", "content": f"""Check the product description against the criteria.
List each criterion it breaks as one short issue; passed is true only if it breaks none.

<criteria>
{CRITERIA}
</criteria>

<facts>{product}</facts>

<description>{text}</description>"""}],
    ).parsed_output
    issues += verdict.issues
    return not issues, issues


text = ask(f"Write a product description for our online shop.\n\n<facts>{product}</facts>", max_tokens=500)
for round_number in range(1, 3):                           # hard limit: 2 rounds
    passed, issues = evaluate(text)
    print(f"ROUND {round_number}: {'PASSED' if passed else 'issues: ' + '; '.join(issues)}\n{text}\n")
    if passed:
        break
    if round_number == 2:
        print("Stopped after 2 rounds: hand it to a person rather than loop again.")
        break
    feedback = "\n".join(f"- {i}" for i in issues)
    text = ask(f"""Rewrite the product description to fix these issues. Reply with the new description only.

<facts>{product}</facts>
<issues>
{feedback}
</issues>
<description>{text}</description>""", max_tokens=500)

ROUND 1: issues: 200 words; the limit is 60; Makes unsupported claim: "so help is always close" overstates what the SOS button (calls two saved numbers) does.; Makes unsupported claim: "gives parents peace of mind" and "reassurance" imply safety the facts don't support.; Unsupported claim: "you can check in whenever you like" implies constant availability not stated in the facts.; Unsupported claim: "can handle splashes, hand-washing, and rainy walks to school" goes beyond "water resistant to 1 metre".
# Nimbus Buddy Kids' Smartwatch

**Stay connected to your child, without handing over a phone.**

The Nimbus Buddy is a smartwatch designed for ages 6 to 12. It gives kids a little independence and gives parents peace of mind.

## Why parents like it

**Know where they are.** Built-in GPS shares your child's location with the Nimbus parents' app, so you can check in whenever you like.

**Help at the press of a button.** If your child ever feels unsafe or needs you, the SOS button calls t

ROUND 2: issues: Claims "no phone needed", which the facts don't support (not stated anywhere in them).
# Nimbus Buddy Kids' Smartwatch

**A smartwatch for ages 6-12, no phone needed.**

- **GPS location:** shared with the parents' app
- **SOS button:** calls two saved numbers
- **Battery:** lasts about 2 days
- **Water resistant:** to 1 metre

**€89**

Add the Nimbus Buddy to your basket today.

Stopped after 2 rounds: hand it to a person rather than loop again.


## 6. An agent

Now Claude chooses the steps. The **environment** is a small in-memory project: a few files and a `run_checks` command that validates a deployment config. The agent gets **generic tools** (list, read, write, run checks), the kind the course recommends: a few flexible tools that combine in many ways, rather than one narrow tool per anticipated situation.

Good agents **inspect the environment**: they read before they change anything, and they check the result after acting (a coding agent runs the tests; a computer-use agent takes a new screenshot). The system prompt asks for that, and the loop has three guards **in code**, because prompts are guidance and these must always hold:

- `write_file` refuses any path outside `config/`, and refuses a JSON file that doesn't parse;
- the loop stops after `MAX_TURNS`;
- after the agent says it's done, code runs the checks itself instead of trusting the claim.

In [6]:
files = {
    "config/README.md": (
        "Deployment config rules:\n"
        "- port: integer from 1024 to 65535\n"
        "- replicas must not exceed max_replicas. max_replicas is set by the platform team: never change it.\n"
        "- log_level: one of debug, info, warning, error\n"
    ),
    "config/app.json": '{"name": "nimbus-api", "port": "8080", "replicas": 4, "max_replicas": 3, "log_level": "verbose"}',
    "src/main.py": "from server import start\n\nstart()\n",
}
ORIGINAL_MAX_REPLICAS = 3


def run_checks():
    try:
        cfg = json.loads(files["config/app.json"])
    except (KeyError, json.JSONDecodeError) as e:
        return f"FAIL config/app.json is missing or not valid JSON: {e}"
    failures = []
    if not isinstance(cfg.get("port"), int) or not 1024 <= cfg["port"] <= 65535:
        failures.append(f"port must be an integer from 1024 to 65535, got {cfg.get('port')!r}")
    if not isinstance(cfg.get("replicas"), int) or cfg["replicas"] > cfg.get("max_replicas", 0):
        failures.append(f"replicas ({cfg.get('replicas')!r}) must not exceed max_replicas ({cfg.get('max_replicas')!r})")
    if cfg.get("log_level") not in {"debug", "info", "warning", "error"}:
        failures.append(f"log_level must be debug, info, warning or error, got {cfg.get('log_level')!r}")
    return "\n".join("FAIL " + f for f in failures) or "PASS all checks"


print(run_checks())       # the starting state: three failures

FAIL port must be an integer from 1024 to 65535, got '8080'
FAIL replicas (4) must not exceed max_replicas (3)
FAIL log_level must be debug, info, warning or error, got 'verbose'


In [7]:
def strict_tool(name, description, properties=None):
    properties = properties or {}
    return {"name": name, "description": description, "strict": True,
            "input_schema": {"type": "object", "properties": properties,
                             "required": list(properties), "additionalProperties": False}}


TOOLS = [
    strict_tool("list_files", "List every file path in the project."),
    strict_tool("read_file", "Read one file. Read a file before changing it, and read README files for rules.",
                {"path": {"type": "string", "description": "File path, e.g. config/app.json"}}),
    strict_tool("write_file", "Replace the whole content of a file under config/. Other paths are refused.",
                {"path": {"type": "string"}, "content": {"type": "string", "description": "The complete new content"}}),
    strict_tool("run_checks", "Validate the deployment config. Run it before and after a change."),
]


class ToolFailure(Exception):
    pass


def execute(name, args):
    if name == "list_files":
        return "\n".join(sorted(files))
    if name == "read_file":
        if args["path"] not in files:
            raise ToolFailure(f"No file {args['path']!r}. Use list_files to see the paths.")
        return files[args["path"]]
    if name == "write_file":
        path, content = args["path"], args["content"]
        if not path.startswith("config/"):                      # guard: scope of changes
            raise ToolFailure(f"Refused: {path!r} is outside config/. Only config files may be changed.")
        if path.endswith(".json"):
            try:
                json.loads(content)
            except json.JSONDecodeError as e:                   # guard: never save broken JSON
                raise ToolFailure(f"Refused: content is not valid JSON ({e}). Nothing was written.")
        files[path] = content
        return f"Wrote {len(content)} characters to {path}."
    if name == "run_checks":
        return run_checks()
    raise ToolFailure(f"Unknown tool {name!r}")


SYSTEM = (
    "You maintain this project's configuration. Work in small steps: look at the files and run the checks "
    "before changing anything, follow the rules in README files, change as little as possible, and run the "
    "checks again after every change. When the checks pass, reply with a two-sentence summary of what you changed."
)
MAX_TURNS = 8


def run_agent(task):
    messages = [{"role": "user", "content": task}]
    for turn in range(1, MAX_TURNS + 1):
        response = client.messages.create(model=model, max_tokens=2000, system=SYSTEM, tools=TOOLS,
                                          output_config={"effort": "medium"}, messages=messages)
        add_assistant_message(messages, response.content)
        if response.stop_reason != "tool_use":
            return text_of(response), turn
        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            try:
                output, is_error = execute(block.name, block.input), False
            except ToolFailure as e:
                output, is_error = str(e), True
            shown = textwrap.shorten(json.dumps(block.input), 90) if block.input else ""
            print(f"turn {turn}: {block.name}({shown}) -> {'ERROR ' if is_error else ''}{textwrap.shorten(output, 90)}")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": output, "is_error": is_error})
        add_user_message(messages, results)
    return f"Stopped after {MAX_TURNS} turns without finishing.", MAX_TURNS


summary, turns = run_agent("The deployment checks are failing. Find out why and fix the config.")
print(f"\nAGENT ({turns} requests): {summary}\n")

# Don't trust the claim: verify the environment in code.
print("independent check:", run_checks())
print("max_replicas unchanged:", json.loads(files["config/app.json"])["max_replicas"] == ORIGINAL_MAX_REPLICAS)
print("final config:", files["config/app.json"])

turn 1: list_files() -> config/README.md config/app.json src/main.py
turn 1: run_checks() -> FAIL port must be an integer from 1024 to 65535, got '8080' FAIL replicas (4) must [...]


turn 2: read_file({"path": "config/README.md"}) -> Deployment config rules: - port: integer from 1024 to 65535 - replicas must not [...]
turn 2: read_file({"path": "config/app.json"}) -> {"name": "nimbus-api", "port": "8080", "replicas": 4, "max_replicas": 3, [...]


turn 3: write_file({"path": "config/app.json", "content": "{\"name\": \"nimbus-api\", \"port\": 8080, [...]) -> Wrote 92 characters to config/app.json.
turn 3: run_checks() -> PASS all checks



AGENT (4 requests): I edited `config/app.json` to fix three problems, and the checks now pass.

- **`port`:** I changed it from the string `"8080"` to the integer `8080`.
- **`replicas`:** I lowered it from 4 to 3 so it doesn't exceed `max_replicas`. I left `max_replicas` at 3, because the README says the platform team owns that value.
- **`log_level`:** I changed it from `verbose`, which isn't allowed, to `debug`. I picked `debug` as the closest match to `verbose`. If you want a different level, set it to `info`, `warning` or `error`.

independent check: PASS all checks
max_replicas unchanged: True
final config: {"name": "nimbus-api", "port": 8080, "replicas": 3, "max_replicas": 3, "log_level": "debug"}


Look at the trace: the agent's path wasn't written anywhere in the code. It chose to list, read (including the README with the rules) and run the checks before writing, and to re-run the checks afterwards. The README rule "never change max_replicas" is what makes it lower `replicas` rather than raise the limit: that's environment inspection paying off. The independent check after the loop is the code's own verification.

One detail in the trace: Claude may send `write_file` and `run_checks` in the **same** turn. This loop runs a turn's tool calls one after another, in order, so the check sees the new file. If your host runs a turn's tool calls concurrently, a "check after writing" step has to land in a later turn.

## 7. Coordinators, subagents and stop conditions

Larger systems combine these pieces. A **coordinator** agent breaks a task down and hands parts to **subagents**, each a separate loop with its own prompt, tools and context window:

```
              coordinator (plans, delegates, merges, decides when done)
             /                  |                    \
   subagent: search      subagent: analyse      subagent: write
   (own context, tools)  (own context, tools)   (own context, tools)
```

- In Claude Code and the Agent SDK, the coordinator starts a subagent with a tool: the exam guide calls it **`Task`**; the current docs call it **`Agent`** (same mechanism).
- A subagent **doesn't inherit** the coordinator's conversation. Everything it needs (goal, constraints, relevant findings so far) must be in the prompt the coordinator writes for it. Missing context in that hand-off is a classic failure.
- Subagents return a **concise result**, not their whole transcript, which keeps the coordinator's context small. Independent subagents can run in parallel (section 4).
- **Stop conditions** live in code: the model ends its turn (`stop_reason` other than `tool_use`), a maximum number of turns, a token or cost budget, a timeout. Don't rely on the model alone to stop.
- **Guards** that must always hold (scope of writes, spending limits, irreversible actions) are enforced in code or hooks, with a human approval step for high-stakes actions; the prompt explains them but isn't what enforces them.

**When not to build an agent:** when the steps are known (use a workflow), when one call with good context answers it, when every run must follow the same audited path, or when latency and cost must be tightly bounded.

**Exam angle (Domain 1).** Scenario questions in this domain usually come down to:

- the simplest architecture that meets the requirement (single call → workflow → agent), and *why* the extra complexity is justified;
- workflows (chain, route, parallelize, evaluate) for known steps; agents for open-ended ones;
- coordinator–subagent hand-offs that pass complete context, and results that come back summarised;
- explicit stop conditions and programmatic guards, rather than more emphatic prompt wording, when a rule must always hold;
- verifying outcomes (checks, tests, re-reading the environment) instead of trusting the agent's own report.

## Recap

- Workflows put the steps in code; agents let Claude pick them. Start simple.
- Chaining: one job per call, code gates in between. Routing: explicit criteria, structured label, code dispatch.
- Parallelization: independent parts concurrently, then aggregate. Evaluator–optimizer: criteria, feedback, and a hard round limit.
- Agents: generic tools, inspect before and after acting, guards and stop conditions in code, verify the result independently.
- Coordinators delegate to subagents (`Task` in the guide, `Agent` today) with complete hand-offs and summarised results.

Next: [2.12 Final assessment and wrap-up](https://anthropic.skilljar.com/claude-with-the-anthropic-api/290899) (in the course; no notebook).